# Reddit Ticker Data Scraper

This notebook scrapes Reddit data based on stock ticker symbols and prepares it for knowledge graph processing.

## Setup Conda Environment

Before running this notebook, set up a dedicated conda environment:

```bash
# Create a new conda environment
conda create -n reddit_scraper python=3.12

# Activate the environment
conda activate reddit_scraper

# Install required packages
pip install praw pandas tqdm
```

Make sure to select this environment as your Jupyter kernel before running the notebook.

In [ ]:
import os
import praw
import pandas as pd
import json
from datetime import datetime
from tqdm import tqdm  # Use standard tqdm instead of notebook version

## Setup Reddit API Credentials

You'll need to create a Reddit application at https://www.reddit.com/prefs/apps to get these credentials.

In [ ]:
# Reddit API credentials - replace with your own
client_id = "YOUR_CLIENT_ID"
client_secret = "YOUR_CLIENT_SECRET"
user_agent = "python:weave.ticker.scraper:v1.0 (by /u/YOUR_USERNAME)"

In [ ]:
# Initialize Reddit API client
reddit = praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent=user_agent
)

## Define Subreddits and Tickers

In [ ]:
# Define subreddits to search
subreddits = [
    "wallstreetbets", 
    "stocks", 
    "investing", 
    "stockmarket"
]

# Define ticker to search for
ticker = "AAPL"  # Example: Apple Inc.

## Functions to Scrape Reddit

In [ ]:
def get_posts_for_ticker(ticker, subreddits, limit=100):
    """
    Scrape posts related to a ticker from specified subreddits.
    
    Parameters
    ----------
    ticker : str
        The stock ticker symbol to search for
    subreddits : list
        List of subreddit names to search in
    limit : int, optional
        Maximum number of posts to retrieve per subreddit
        
    Returns
    -------
    list
        List of dictionaries containing post data
    """
    all_posts = []
    
    for subreddit_name in tqdm(subreddits, desc="Searching subreddits"):
        subreddit = reddit.subreddit(subreddit_name)
        
        # Search for posts containing the ticker
        for post in tqdm(subreddit.search(ticker, limit=limit), 
                         desc=f"Fetching posts from r/{subreddit_name}", 
                         total=limit):
            # Get top comments
            post.comments.replace_more(limit=0)  # Remove comment stubs
            top_comments = []
            for comment in post.comments[:10]:  # Get top 10 comments
                top_comments.append({
                    "body": comment.body,
                    "score": comment.score,
                    "author": str(comment.author),
                    "created_utc": datetime.fromtimestamp(comment.created_utc).isoformat()
                })
            
            # Create Reddit permalink URL (actual URL to the post)
            reddit_url = f"https://www.reddit.com{post.permalink}"
            
            # Create post data dictionary
            post_data = {
                "id": post.id,
                "title": post.title,
                "selftext": post.selftext,
                "url": post.url,  # Original URL (might be external link)
                "reddit_url": reddit_url,  # Actual Reddit post URL
                "score": post.score,
                "upvote_ratio": post.upvote_ratio,
                "num_comments": post.num_comments,
                "created_utc": datetime.fromtimestamp(post.created_utc).isoformat(),
                "subreddit": subreddit_name,
                "author": str(post.author),
                "top_comments": top_comments,
                "ticker": ticker
            }
            
            all_posts.append(post_data)
    
    return all_posts

In [ ]:
def save_to_jsonl(data, filename):
    """
    Save data to JSONL format, compatible with the existing pipeline.
    
    Parameters
    ----------
    data : list
        List of dictionaries to save
    filename : str
        Output filename
    """
    with open(filename, 'w', encoding='utf-8') as f:
        for item in data:
            # Combine title, post text, and top comments into a single "content" field
            # to match the expected format for article processing
            combined_text = f"{item['title']}\n\n{item['selftext']}"
            
            # Add top comments
            if item['top_comments']:
                combined_text += "\n\nTop Comments:\n"
                for i, comment in enumerate(item['top_comments'], 1):
                    combined_text += f"\n{i}. {comment['body']}"
            
            # Create a document in the format expected by the existing pipeline
            # Use the reddit_url (permalink) instead of the original url which might be external
            document = {
                "url": item['reddit_url'],  # Use the actual Reddit post URL
                "title": item['title'],
                "date": item['created_utc'],
                "content": combined_text,
                "source": f"reddit-{item['subreddit']}",
                "ticker": item['ticker']
            }
            
            f.write(json.dumps(document) + '\n')

## Scrape Reddit Data for Ticker

In [ ]:
# Scrape posts for the specified ticker
posts = get_posts_for_ticker(ticker, subreddits, limit=50)

In [ ]:
# Preview the data
print(f"Total posts scraped: {len(posts)}")
if posts:
    # Create a DataFrame for easier inspection
    df = pd.DataFrame(posts)
    df[['id', 'title', 'score', 'num_comments', 'subreddit', 'created_utc']].head()

## Save to JSONL for Pipeline Processing

In [ ]:
# Save the data to a JSONL file
output_filename = f"reddit_{ticker.lower()}.jsonl"
save_to_jsonl(posts, output_filename)
print(f"Data saved to {output_filename}")

## Next Steps: Processing with BAML Pipeline

After saving the Reddit data to a JSONL file, follow these steps to process it with the BAML pipeline:

1. Generate the BAML client code:
```bash
baml-cli generate
```

2. Process the Reddit data through the LLM extraction pipeline:
```bash
poetry run abzu process articles reddit
```

3. Build the knowledge graph from the processed Reddit data:
```bash
poetry run abzu process kg raw --input data/processed_reddit.jsonl --output data/reddit_knowledge_graph
```

4. Refine the knowledge graph:
```bash
poetry run abzu process kg refine --input data/reddit_knowledge_graph --output data/refined_reddit_knowledge_graph
```

For convenience, you can view all these steps with:
```bash
poetry run abzu steps reddit
```

After processing, the knowledge graph data will be available in the `data/refined_reddit_knowledge_graph` directory for visualization.

## Running the Pipeline

To execute the complete Reddit processing pipeline, you can run each step individually, or use the following bash script:

```bash
#!/bin/bash
set -e

# Generate BAML client code
echo "Generating BAML client code..."
baml-cli generate

# Process Reddit data through LLM extraction
echo "Processing Reddit data..."
poetry run abzu process articles reddit

# Build knowledge graph from processed data
echo "Building knowledge graph..."
poetry run abzu process kg raw --input data/processed_reddit.jsonl --output data/reddit_knowledge_graph

# Refine knowledge graph
echo "Refining knowledge graph..."
poetry run abzu process kg refine --input data/reddit_knowledge_graph --output data/refined_reddit_knowledge_graph

echo "Pipeline completed successfully!"
```

Save this script to a file (e.g., `process_reddit.sh`), make it executable with `chmod +x process_reddit.sh`, and run it with `./process_reddit.sh`.